# ORCA-X — Google Colab GPU Training

This notebook runs the current canonical ORCA-X XGBoost production training pipeline using a Colab GPU. It preserves the completed ML refinements: real historical Open-Meteo data, six coastal locations, 6-hour forward target construction, point-in-time features, class weighting, temporal validation, Digha spatial holdout, and production model metadata.

**Before running:** Runtime → Change runtime type → select T4/L4 GPU.

In [1]:
REPO_URL = 'https://github.com/Sayan260106/HackHeritage.git'
REPO_REF = 'feat/colab-gpu-ml'  # Change to main after this branch is merged.
REPO_DIR = '/content/HackHeritage'
START_DATE = '2020-01-01'
END_DATE = '2025-12-31'
LOCATIONS = ['digha_wb', 'paradip_od', 'vizag_ap', 'chennai_tn', 'goa', 'kochi_kl']
import os
os.environ['ORCA_X_DEVICE'] = 'cuda'
os.environ['ORCA_X_N_JOBS'] = '2'
print('ORCA_X_DEVICE =', os.environ['ORCA_X_DEVICE'])
print('ORCA_X_N_JOBS =', os.environ['ORCA_X_N_JOBS'])

ORCA_X_DEVICE = cuda
ORCA_X_N_JOBS = 2


In [2]:
!rm -rf "$REPO_DIR"
!git clone --depth 1 --branch "$REPO_REF" "$REPO_URL" "$REPO_DIR"
%cd "$REPO_DIR"
!git rev-parse --short HEAD
!git branch --show-current

Cloning into '/content/HackHeritage'...
remote: Enumerating objects: 133, done.
remote: Counting objects: 100% (133/133), done.
remote: Compressing objects: 100% (114/114), done.
remote: Total 133 (delta 11), reused 102 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (133/133), 1.11 MiB | 6.70 MiB/s, done.
Resolving deltas: 100% (11/11), done.
/content/HackHeritage
96dcb1e
feat/colab-gpu-ml


In [3]:
!python -m pip install -q --upgrade pip
!python -m pip install -q -r ml/requirements-colab.txt
!nvidia-smi
import xgboost as xgb
print('XGBoost version:', xgb.__version__)
probe = xgb.XGBClassifier(n_estimators=2, max_depth=2, tree_method='hist', device='cuda', objective='multi:softprob', num_class=4)
print('Configured XGBoost device:', probe.get_params()['device'])

/bin/bash: line 1: nvidia-smi: command not found
XGBoost version: 3.4.1
Configured XGBoost device: cuda


## Download and prepare the real historical dataset

The raw dataset does not need to be committed to Git. Colab downloads the same Open-Meteo historical weather + marine sources used by ORCA-X and rebuilds the canonical parquet.

In [4]:
location_args = ' '.join(LOCATIONS)
!python ml/src/download_historical_marine.py --start "$START_DATE" --end "$END_DATE" --locations $location_args
!python ml/src/prepare_dataset.py

GET weather digha_wb
GET marine  digha_wb
GET weather paradip_od
GET marine  paradip_od
GET weather vizag_ap
GET marine  vizag_ap
GET weather chennai_tn
GET marine  chennai_tn
GET weather goa
GET marine  goa
GET weather kochi_kl
GET marine  kochi_kl
Historical Open-Meteo marine/weather download complete.
Raw directory: /content/HackHeritage/ml/data/raw/open_meteo
READ digha_wb (Digha Coast)
  rows: 52,608
READ paradip_od (Paradip Coast)
  rows: 52,608
READ vizag_ap (Visakhapatnam Coast)
  rows: 52,608
READ chennai_tn (Chennai Coast)
  rows: 52,608
READ goa (Goa Coast)
  rows: 52,608
READ kochi_kl (Kochi Coast)
  rows: 52,608
ORCA-X HISTORICAL DATASET READY
Rows: 315,648
Locations: 6
Risk distribution:
risk_label
EXTREME      15105
HIGH        112822
LOW         169164
MODERATE     18557
Name: count, dtype: int64
Model feature dtypes:
wind_speed_kts               float64
wind_gust_kts                float64
wave_height_m                float64
wave_period_s                float64
swell_

In [5]:
import pandas as pd
df = pd.read_parquet('ml/data/processed/orca_historical_marine_risk.parquet')
print('Rows:', f'{len(df):,}')
print('Locations:', df['location_id'].nunique())
print(df['location_id'].value_counts().sort_index())
print('\nRisk distribution:')
print(df['risk_label'].value_counts().sort_index())

Rows: 315,648
Locations: 6
location_id
chennai_tn    52608
digha_wb      52608
goa           52608
kochi_kl      52608
paradip_od    52608
vizag_ap      52608
Name: count, dtype: int64

Risk distribution:
risk_label
EXTREME      15105
HIGH        112822
LOW         169164
MODERATE     18557
Name: count, dtype: int64


## Train the canonical production model on GPU

`ml/src/train.py` reads `ORCA_X_DEVICE`. With `cuda`, XGBoost uses the Colab GPU. The forward-target construction and Digha holdout remain inside the canonical training script, so the Colab run does not change the leakage controls.

In [ ]:
!python ml/src/train.py

XGBoost execution device: cuda
XGBoost n_jobs: 2
GPU mode enabled. In Google Colab verify Runtime > Change runtime type > T4/L4 GPU before running.
Dataset rows after forward-target construction: 315,612; locations: 6
Prediction horizon: +6h
Risk policy: orca-operational-risk-v3-sustained-wind-sea-state
Feature count: 44 (17 base + point-in-time engineered features)
Missing percentage by feature:
wind_speed_kts                        0.00
wind_gust_kts                         0.00
wave_height_m                        29.16
wave_period_s                        29.21
swell_height_m                       29.16
swell_period_s                       29.21
wind_direction_deg                    0.00
wave_direction_deg                   29.16
swell_direction_deg                  29.16
air_pressure_hpa                      0.00
air_temperature_c                     0.00
sea_surface_temperature_c            48.83
precipitation_mm                      0.00
latitude                              0.0

In [ ]:
import json
from pathlib import Path
metadata = json.loads(Path('ml/models/orca_xgb_risk_metadata.json').read_text())
print('Model version:', metadata['model_version'])
print('Dataset version:', metadata['dataset_version'])
print('Prediction horizon:', metadata['prediction_horizon_hours'], 'hours')
print('Training device:', metadata['training_device'])
print('Feature count:', metadata['feature_count'])
print('Digha excluded from training:', metadata['digha_excluded_from_training'])
print('\nTemporal validation:')
print(json.dumps(metadata['evaluation']['temporal'], indent=2))
print('\nDigha spatial holdout:')
print(json.dumps(metadata['evaluation']['digha_spatial_holdout'], indent=2))

In [ ]:
import zipfile
bundle = '/content/orca_x_model_bundle.zip'
with zipfile.ZipFile(bundle, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    z.write('ml/models/orca_xgb_risk.json', 'orca_xgb_risk.json')
    z.write('ml/models/orca_xgb_risk_metadata.json', 'orca_xgb_risk_metadata.json')
print('Created:', bundle)
print('Download it from the Colab file browser and put both files in HackHeritage/ml/models/.')

## Refinements 20–26

The later refinements are retained as leakage/provenance, point-in-time availability, robustness, reliability and uncertainty benchmarks. They are read-only evaluation/audit work and are not silently substituted for the canonical production model. The production inference contract remains point-in-time features only.